In [1]:
from openai import OpenAI
import json
import os
import sqlite3
from with_bdd_prompts import *
from with_bdd_requests import *


In [11]:
with open("../../env/keys.json", "r", encoding="utf-8") as f:
    keys = json.load(f)

key = keys["OPENAI_SNCF"]
os.environ["OPENAI_API_KEY"] = key
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

In [5]:
def get_incidents_voiture():
    
    conn = sqlite3.connect("/Users/margauxlanglois/Documents/SNCF/s2025p2-mobile-app-incidents/arborescence_analyse/openAI/indicents.db")
    cursor = conn.cursor()
    return conn, cursor 

conn, cursor = get_incidents_voiture()
cursor.execute("SELECT * FROM id;")
print(cursor.fetchall())



OperationalError: no such table: id

In [21]:
def get_incidents_voiture():
    
    conn = sqlite3.connect("/Users/margauxlanglois/Documents/SNCF/s2025p2-mobile-app-incidents/arborescence_analyse/DBs/incidents.db")
    cursor = conn.cursor()
    return conn, cursor 

conn, cursor = get_incidents_voiture()
cursor.execute("""
        SELECT localisation, categorie, organe, precision_n2, precision_n3, sous_organe, defaillance
        FROM DASYE_eau_incidents
        WHERE rames LIKE '%' || ? || '%';
    """, ("R1", ))
response = cursor.fetchall()
print(response)

[('Place', 'Equipements SECURITE', 'Vitre latérale', '', '', '', 'Bruyante, vibrations, sifflement, embuée'), ('Place', 'Equipements SECURITE', 'Vitre latérale', '', '', '', "Problème d'étanchéité, Infiltration d'eau"), ('Place', 'Equipements SECURITE', 'Vitre latérale', '', '', '', 'Rayée, gravée'), ('Place', 'Equipements SECURITE', 'Vitre latérale', '', '', '', 'Verre extérieur fissuré, impact, mosaïqué'), ('Place', 'Equipements SECURITE', 'Vitre latérale', '', '', '', 'Verre intérieur mosaïqué ET verre extérieur intact'), ('Place', 'Eclairage', 'Eclairage latéral (au dessus des vitres)', '', '', '', 'Ne fonctionne pas, manquant, cassé'), ('Place', 'Eclairage', 'Eclairage sol (balisage)', '', '', '', 'Ne fonctionne pas, manquant, cassé'), ('Place', 'Pack inoui', 'Film Baie', '', '', 'Laissez vs rever', 'Abîmé, dégradé, manquant'), ('Place', 'Pack inoui', 'Film Baie', '', '', 'Siège avec vue', 'Abîmé, dégradé, manquant'), ('Place', 'Pack inoui', 'Film Baie', '', '', 'NoFilter', 'Abîmé

In [ ]:
def prompt_openai_objects(list_objects, message):

    messages = [
            {"role": "system", "content": "Tu es un assistant chargé d'analyser des transcriptions audio d'agents SNCF pour identifier l'objet concerné par l'incident."},
            {"role": "user", "content": f"""Voici les objets possibles :
    {list_objects}

    Transcription :
    {message}


    ⚠️ Réponds uniquement par l'incident qui se rapproche le plus du signalement.
    Si tu n'es pas sûr, écris : Je ne sais pas."""}
        ]

    return messages

In [19]:

def ask_openai(messages, tools=None):
    response = client.chat.completions.create(
        model="gpt-4.1",  # ou "gpt-4.1" si tu l’utilises
        messages=messages,
    )
    print(response.choices[0].message.content)

In [ ]:
message1 = "Le sol est mouillé place 54"

In [26]:
message = "Il y a un problème avec la serrure du local"
print(ask_openai(prompt_openai_objects(response, message)))

('Sanitaire', 'Porte local', 'Serrure', '', '', '', 'Problème de verrouillage')
None
